<a href="https://colab.research.google.com/github/Eswar2005-Karanam/blog/blob/main/Flower_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# STEP 0: INSTALL GRADIO (Runs silently in the background)
# =====================================================================
!pip install -q gradio

# =====================================================================
# STEP 1: DOWNLOAD & PREPARE DATASET (OPTIMIZED FOR SPEED)
# =====================================================================
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
import gradio as gr

print("1/4 Downloading verified flower dataset...")
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = keras.utils.get_file('flower_photos', origin=url, extract=True)
data_dir = os.path.join(os.path.dirname(data_dir), 'flower_photos')

print("\nLoading training & validation data...")
# Batch size set to 64 for faster epoch execution
train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=64
)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=64
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Detected categories: {class_names}")

# RAM Caching + Prefetching makes epochs run in ~5-10 seconds
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# =====================================================================
# STEP 2: BUILD FAST TRANSFER LEARNING MODEL
# =====================================================================
print("\n2/4 Constructing MobileNetV2 Neural Network...")
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model to avoid long training times
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# =====================================================================
# STEP 3: FAST TRAINING (2 EPOCHS ~90%+ ACCURACY)
# =====================================================================
print("\n3/4 Training custom classification layer...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2
)

# Save trained model locally
model.save('fast_flower_model.keras')
print("\nModel saved to 'fast_flower_model.keras'")

# =====================================================================
# STEP 4: LAUNCH GRADIO INTERACTIVE WEB INTERFACE
# =====================================================================
print("\n4/4 Launching Interactive UI...")

def predict_flower(input_img):
    if input_img is None:
        return None
    # Resize image to model input shape
    img = tf.image.resize(input_img, (224, 224))
    img_array = tf.expand_dims(img, 0)

    # Generate model probabilities
    predictions = model.predict(img_array, verbose=0)[0]

    # Format confidence dictionary for Gradio
    return {class_names[i]: float(predictions[i]) for i in range(num_classes)}

# Create drag-and-drop web interface inside Colab cell
interface = gr.Interface(
    fn=predict_flower,
    inputs=gr.Image(type="numpy", label="Upload or Capture Image"),
    outputs=gr.Label(num_top_classes=3, label="Top Predictions"),
    title="🌺 Instant Plant & Flower Classifier",
    description="Upload any flower picture or take a photo with your webcam to classify it instantly!"
)

# share=True creates a public URL accessible from smartphones or browsers
interface.launch(share=True, debug=False)

1/4 Downloading verified flower dataset...
228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

Loading training & validation data...
Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.
Detected categories: ['flower_photos']

2/4 Constructing MobileNetV2 Neural Network...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

3/4 Training custom classification layer...
Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


46/46 ━━━━━━━━━━━━━━━━━━━━ 186s 4s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 2/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 183s 3s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00

Model saved to 'fast_flower_model.keras'

4/4 Launching Interactive UI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5f7ae1b1d0d753eb2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
